# LAB 07 — Transformer Classification Experiments

Notebook này triển khai Bài 19–23, 25 và 29 theo W7.pdf. Mọi metrics/đồ thị được tạo từ model thực sự chạy. Các phần nhận xét cá nhân để sinh viên tự hoàn thiện.

## 1. Imports & Environment

Import TransformerBlock tự cài ở Bài 17; chạy regression tests để chắc thay đổi mask/inspection vẫn giữ hoạt động cũ.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch
from IPython.display import display

LAB_DIR = Path.cwd()
if not (LAB_DIR / "W7.pdf").exists():
    LAB_DIR = (LAB_DIR / "Lab7").resolve()
sys.path.insert(0, str(LAB_DIR))

from transformer_block import TransformerBlock, run_tests as test_transformer_block
from transformer_experiments import (
    ExperimentRunner, SEED, SPLIT_SEED, D_MODEL, D_FF, BATCH_SIZE,
    LEARNING_RATE, EPOCHS, MAX_LENGTH, MAX_LENGTHS,
)

test_transformer_block()

## 2. Configuration & Reproducibility

`full` dùng toàn bộ split LAB5 và 5 epochs. `smoke` chỉ kiểm tra pipeline trên các tập con xác định (2.048 train, 512 validation, 1.024 test) với 1 epoch; không dùng smoke metrics thay cho kết quả full.

In [ ]:
RUN_MODE = "full"  # đổi thành "smoke" để kiểm tra nhanh pipeline
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {
    "seed": SEED,
    "split_seed_lab5": SPLIT_SEED,
    "d_model": D_MODEL,
    "d_ff": D_FF,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS if RUN_MODE == "full" else 1,
    "max_length_default": MAX_LENGTH,
    "max_lengths": list(MAX_LENGTHS),
    "transformer_blocks": 1,
    "run_mode": RUN_MODE,
    "device": DEVICE,
}
print(CONFIG)
print("CUDA available:", torch.cuda.is_available())

## 3. Dataset Loading — IMDb, đúng split LAB5

Dùng split IMDb có nhãn `train` và `test`; không dùng `unsupervised`. IMDb train được chia 80/20 stratified với `random_state=2026` như LAB5; test gốc giữ nguyên. Model seed là 42.

In [ ]:
runner = ExperimentRunner(LAB_DIR, device=DEVICE, run_mode=RUN_MODE)
ENVIRONMENT = runner.environment_summary()
display(pd.Series(ENVIRONMENT, name="Value").to_frame())
print("Train/validation/test sizes:", len(runner.train_ids), len(runner.val_ids), len(runner.test_labels))
print("Vocabulary size, including PAD/UNK:", len(runner.vocab))
print("Label mapping: 0=Negative, 1=Positive; positive class for F1=1")

## 4. Tokenizer & Vocabulary

Tái sử dụng cách làm sạch/tokenizer regex LAB5. Vocabulary tối đa 20.000 mục, min frequency 2, chỉ dựng từ train. PAD và UNK có ID riêng. Review được truncate bên phải theo max length và padding động theo batch.

In [ ]:
print("Tokenizer: HTML unescape + lowercase + strip HTML + [a-z0-9]+(?:'[a-z]+)?")
print("PAD_ID:", 0, "UNK_ID:", 1)
print("Vocabulary sample:", list(runner.vocab.items())[:12])
for split, ids, labels in (
    ("train", runner.train_ids, runner.train_labels),
    ("validation", runner.val_ids, runner.val_labels),
):
    print(split, "samples:", len(ids), "labels:", pd.Series(labels).value_counts().sort_index().to_dict())

## 5. Shared Training & Evaluation Utilities

`ExperimentRunner` dùng chung loaders, optimizer Adam, CrossEntropyLoss, metrics binary (positive=1), early checkpoint selection theo Validation F1 và loss trung bình theo số samples. Test chỉ được gọi ở phần final evaluation.

In [ ]:
print("Batch size:", BATCH_SIZE, "| LR:", LEARNING_RATE, "| epochs:", CONFIG["epochs"])
print("Attention masks exclude PAD keys; pooling excludes PAD positions.")
print("Train shuffling restarts with the same seed for each independent model.")

## 6. Transformer Classifier — Bài 23

Pipeline: Embedding → sinusoidal positional encoding → một TransformerBlock tự cài → masked mean pooling → Linear classifier. Classifier trả logits; xác suất chỉ tính khi cần prediction. Có thể bật learned `[CLS]` vector ở Bài 25.

In [ ]:
print("TransformerBlock source:", (LAB_DIR / "transformer_block.py").resolve())
print("Padding mask is applied to attention keys and to mean pooling.")
print("One block; d_model=128, d_ff=512, dropout=0.1.")

## 7. Baseline Classification & 8. Number of Attention Heads — Bài 23 và 19

Mỗi số head tạo model mới, seed/optimizer/data/batch/epochs/length/pooling giữ nguyên. Bảng dùng Validation metrics để so sánh và chọn checkpoint; cột parameters là toàn classifier và có thêm số riêng của TransformerBlock.

In [ ]:
heads_results = runner.run_heads()
display(heads_results.style.format({
    "Training time (s)": "{:.2f}", "Validation accuracy": "{:.4f}",
    "Validation F1": "{:.4f}", "Validation loss at best-F1 checkpoint": "{:.4f}",
}))

### Nhận xét và trả lời câu hỏi — Bài 19

Với `d_model=128` cố định, `d_k` lần lượt là 128, 64, 32 và 16 khi head count là 1, 2, 4, 8. Số tham số của cả bốn classifier bằng 2.758.530 vì implementation giữ tổng chiều các head bằng `n_heads × d_k = 128`; các phép chiếu Q/K/V và output projection vì vậy giữ kích thước tương đương. Trên validation, F1 lần lượt là 0,8361; 0,8384; 0,8333; 0,8339. Accuracy cao nhất thuộc 2 heads (0,8378), còn 8 heads có recall cao nhất (0,8784) nhưng precision thấp nhất (0,7936). Thời gian đo được dao động 155,39–214,80 giây. Do đó tăng heads không làm representation tổng thể lớn hơn trong cấu hình này và cũng không bảo đảm F1 tăng. Kết quả chỉ phản ánh một lần chạy với seed/cấu hình đã ghi; thời gian phụ thuộc phần cứng và tải máy.

## 9. Attention Head Diversity — Bài 20

Lấy attention weights từ model 4-head đã train, ở `eval()` mode. Heatmaps dùng chung scale; thống kê entropy, max weight và cosine similarity chỉ mô tả phân phối, không tự chứng minh ý nghĩa ngôn ngữ.

In [ ]:
head_stats, head_similarity = runner.run_head_diversity()
display(head_stats.style.format({
    "Mean attention entropy": "{:.4f}", "Mean maximum weight": "{:.4f}"
}))
display(head_similarity.style.format("{:.4f}"))

### Nhận xét và trả lời câu hỏi — Bài 20

Heatmap được tạo từ một review validation cụ thể, nên các kết luận chỉ áp dụng cho mẫu đó. Trong bốn heads, Head 1 có entropy trung bình cao nhất (1,4856) và maximum weight thấp nhất (0,4776), phù hợp với phân bố tương đối dàn trải. Head 2 có entropy thấp nhất (0,9608) và maximum weight cao nhất (0,6946), tức phân bố tập trung hơn. Mean pairwise similarity nằm trong khoảng 0,1607–0,3024; đây là mức độ giống nhau không cao trong thống kê được dùng, và hình cho thấy các pattern khác nhau. Dữ liệu này gợi ý các heads không hoàn toàn trùng nhau ở mẫu quan sát, nhưng chưa đủ để kết luận chúng có vai trò ngữ pháp/ngữ nghĩa riêng hay không có redundancy trên toàn corpus. Attention weight mô tả phân phối truy vấn tới key tại lớp được đo; nó không trực tiếp chứng minh token nào quyết định sentiment. Muốn kết luận tổng quát cần đo trên nhiều review và nhiều lớp.

## 10. Transformer vs LSTM — Bài 21

LSTM dùng packed sequences để bỏ padding; hai model dùng chung data split, vocabulary, embedding/hidden size 128, batch, optimizer, LR, epochs và max length 128. Số parameters không ép bằng nhau. Bảng dưới là Validation; test sẽ được báo cáo một lần ở phần cuối.

In [ ]:
comparison_validation = runner.run_lstm()
display(comparison_validation.style.format({
    "Training time (s)": "{:.2f}", "Validation accuracy": "{:.4f}",
    "Validation F1": "{:.4f}", "Validation loss at best-F1 checkpoint": "{:.4f}",
}))

### Nhận xét và trả lời câu hỏi — Bài 21

Trên validation, LSTM đạt Accuracy 0,8298 và F1 0,8347; Transformer 4-head đạt lần lượt 0,8300 và 0,8333. LSTM có recall cao hơn (0,8596 so với 0,8496), còn Transformer có precision cao hơn (0,8176 so với 0,8113). LSTM có 2.692.354 tham số và thời gian huấn luyện 298,20 giây; Transformer có 2.758.530 tham số và mất 206,40 giây ở lần chạy được ghi. Trên test, LSTM F1 là 0,8189 và Transformer là 0,8085, nên thứ tự validation không đồng nhất với test. Recurrence của LSTM truyền trạng thái theo thứ tự token; self-attention cho phép token tương tác trực tiếp trong block, còn positional encoding cung cấp thông tin vị trí. LSTM dùng hidden state cuối, Transformer dùng masked mean pooling trong so sánh này. Vì pooling và cơ chế tuần tự khác nhau, chênh lệch không thể quy về một thành phần duy nhất; hơn nữa đây là một split và một cấu hình.

## 11. Sequence Length — Bài 22

So sánh max_length 32/64/128/256. Length 128 tái sử dụng đúng model 4-head đã train ở Bài 19. Tokens retained được đo trên train; attention memory là ước lượng cho **một** score/weight matrix `(batch, heads, T, T)`, không phải tổng bộ nhớ training.

In [ ]:
length_results = runner.run_sequence_lengths()
display(length_results.style.format({
    "Mean tokens retained/sample": "{:.2f}", "Train tokens retained (%)": "{:.2f}%",
    "Training time (s)": "{:.2f}", "Validation F1": "{:.4f}",
    "Validation accuracy": "{:.4f}",
}))

### Nhận xét và trả lời câu hỏi — Bài 22

Khi max length tăng 32→64→128→256, tỷ lệ token train được giữ tăng 13,62%→26,97%→50,89%→76,44%; validation F1 tăng tương ứng 0,7447→0,7992→0,8333→0,8654. Thời gian train tăng từ 55,44 lên 462,90 giây. Kích thước lý thuyết của một attention matrix theo batch/head tăng từ 524.288 bytes ở length 32 lên 33.554.432 bytes ở 256, đúng hệ số 64 do số phần tử tỷ lệ `T²`. Đây chỉ là kích thước một ma trận float32, không phải tổng bộ nhớ GPU hay RAM của cả quá trình. Dữ liệu này cho thấy ở corpus và các lần chạy này, giữ thêm token đi cùng F1 validation cao hơn nhưng chi phí thời gian/memory matrix tăng đáng kể. Không thể kết luận length lớn luôn tốt hơn: các cấu hình này không đo mọi độ dài, thời gian có thể chịu ảnh hưởng tải máy, và kết quả là validation chứ không phải đánh giá test cho từng length.

## 12. Pooling Strategy — Bài 25

Mean pooling loại padding. CLS-style dùng một learned `[CLS]` vector đặt ở đầu sequence; positional encoding/mask được dịch và cập nhật tương ứng. Chỉ thay pooling và token cần thiết.

In [ ]:
pooling_results = runner.run_pooling()
display(pooling_results.style.format({
    "Training time (s)": "{:.2f}", "Validation accuracy": "{:.4f}",
    "Validation F1": "{:.4f}",
}))

### Nhận xét và trả lời câu hỏi — Bài 25

Với cùng cấu hình 4 heads và max length 128, masked mean pooling đạt validation Accuracy 0,8300, F1 0,8333, có 2.758.530 tham số và thời gian 206,40 giây. CLS-style learned pooling đạt Accuracy 0,8334, F1 0,8359, có 2.758.658 tham số và thời gian 158,48 giây. Chênh lệch F1 là 0,0027 (khoảng 0,27 điểm phần trăm); model CLS có thêm 128 tham số tương ứng vector CLS học được. Ở kết quả này CLS nhỉnh hơn một chút về Accuracy/F1 và thời gian ghi nhận thấp hơn, nhưng không nên xem thời gian của một lần chạy là bằng chứng kiến trúc luôn nhanh hơn. Mean pooling tổng hợp các biểu diễn token không padding; CLS pooling dùng biểu diễn token đặc biệt ở đầu. Kết quả chỉ dùng validation và một lần chạy, nên cần lặp lại với nhiều seed hoặc đánh giá test cuối cùng trước khi khẳng định khác biệt ổn định.

## 13. Learning Curves — Bài 29

Loss được lưu từng epoch, train loss theo train samples và validation loss theo validation samples. Biểu đồ không tự gán kết luận overfitting/underfitting.

In [ ]:
learning_history = runner.save_learning_curves()
display(learning_history.style.format({
    "train_loss": "{:.4f}", "validation_loss": "{:.4f}", "validation_f1": "{:.4f}"
}))

### Nhận xét và trả lời câu hỏi — Bài 29

Ở Transformer 4-head, train loss giảm liên tục từ 0,5482 xuống 0,1281 trong năm epoch. Validation loss thấp nhất ở epoch 3 (0,3849), sau đó tăng lên 0,4348 và 0,5323; validation F1 cao nhất lại ở epoch 5 (0,8333). Vì checkpoint được chọn theo validation F1, loss 0,5323 là loss tại checkpoint được chọn, không phải “best validation loss”. Đây là lý do bảng phải ghi rõ tiêu chí checkpoint. Với LSTM, train loss giảm 0,6866→0,2274; validation loss thấp nhất và F1 cao nhất đều ở epoch 4 (0,3958 và 0,8347), rồi validation loss tăng lên 0,4111 và F1 giảm nhẹ còn 0,8342 ở epoch 5. Hai đường cong cho thấy khoảng cách train/validation mở rộng ở cuối một số epoch, đặc biệt ở Transformer. Đây là dấu hiệu cần theo dõi, nhưng năm epoch và một lần chạy chưa đủ để khẳng định nguyên nhân hoặc mức độ overfitting.

## 14. Confusion Matrix & Final Test Evaluation

Sau khi cấu hình chính được chọn qua Validation, đánh giá Transformer 4-head mean baseline và LSTM trên IMDb test split gốc (hoặc test subset nếu RUN_MODE=smoke). Confusion matrix tương ứng Transformer; tổng ô phải bằng số test samples.

In [ ]:
test_results, test_labels, transformer_test_predictions = runner.final_test_evaluation()
display(test_results.style.format({
    "Training time (s)": "{:.2f}", "Accuracy": "{:.4f}",
    "Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}",
}))
print("Evaluated test samples:", len(test_labels))

## 15. Application — Năm loại input

Đây là kiểm tra hành vi trên câu thủ công, không thay evaluation test. Xác suất lấy softmax từ logits của model đã train.

In [ ]:
example_predictions = runner.predict_examples()
display(example_predictions.style.format({"P(Negative)": "{:.4f}", "P(Positive)": "{:.4f}"}))

### Nhận xét và trả lời câu hỏi — Bài 23

Trên IMDb test, Transformer 4-head đạt Accuracy 0,8071 và F1 0,8085; confusion matrix có TN=10.001, FP=2.499, FN=2.323, TP=10.177. Năm câu kiểm tra thủ công cho output: câu ngắn “I loved this movie.” → Positive (P=0,999976); câu dài có nhiều ý khen/chê → Negative (0,999017); “I did not enjoy this film.” → Positive (0,978495); câu tương phản “good beginning but ... terrible” → Negative (0,993031); câu mơ hồ “not bad ... not especially memorable” → Negative (0,998063). Đây là input tự tạo, không có nhãn chuẩn đi kèm, vì vậy chỉ câu phủ định có ý định sentiment rõ mới cho thấy prediction trái với cách đọc thông thường; không tính accuracy trên năm câu này. Kết quả minh họa model có thể xử lý một số câu tương phản nhưng chưa xử lý phủ định nhất quán. Probability cao là độ tự tin của model, không đảm bảo câu trả lời đúng.

## 16. Export Results

`results.csv` có các dòng riêng cho từng experiment, validation/test split và epoch loss. Trường không áp dụng để trống; kết quả smoke được đánh dấu `run_mode=smoke`.

In [ ]:
results = runner.export_results()
display(results.groupby(["experiment", "eval_split"], dropna=False).size().rename("rows").to_frame())
print("Results path:", LAB_DIR / "results.csv")
print("Images:", LAB_DIR / "attention_heads.png", LAB_DIR / "learning_curves.png", LAB_DIR / "confusion_matrix.png")

## 17. Student Interpretation TODOs

- Bài 19: Tăng head có tăng representation dimension không? `d_k` đổi ra sao? Vì sao F1 có thể không tăng?
- Bài 20: Head nào tập trung/phân tán? Các ma trận có giống nhau không? Evidence nào hỗ trợ nhận xét?
- Bài 21: Khác biệt architecture nào có thể liên quan đến kết quả; sequential và parallel processing khác gì?
- Bài 22: Vì sao không chỉ tăng sequence length? T² và thời gian thực tế thay đổi ra sao?
- Bài 25: Pooling ảnh hưởng gì tới representation và kết quả trên split này?
- Bài 29: Quan sát convergence/instability/overfitting từ từng đường loss; không kết luận khi chưa có bằng chứng.

## 18. AI Assistance Statement — sinh viên rà soát và hoàn thiện

```text
Tool: OpenAI Codex
Purpose: hỗ trợ boilerplate, debugging, visualization và chạy/kiểm tra experiments.
Generated: [liệt kê đúng phần code/biểu đồ được AI hỗ trợ]
Modified: [ghi phần em đã tự sửa]
Verified: [ghi những phép kiểm tra em đã tự xác nhận]
```

Notebook không viết thay phần interpretation, error analysis hoặc reflection cá nhân.

## 19. Error Analysis — Bài 26

Sáu review bên dưới được chọn từ dự đoán thực tế trên IMDb test bằng checkpoint 4-head. Cell chỉ đọc artifacts đã lưu và không tự huấn luyện lại model. Phân tích chi tiết, trích dẫn, attention evidence và giới hạn nằm trong [`error_analysis.md`](error_analysis.md).

In [ ]:
from IPython.display import Image, display
from error_analysis import run_error_analysis

# Dùng model hiện có trong runner hoặc checkpoint; không train lại trong cell này.
error_samples, error_attention, error_summary = run_error_analysis(
    runner=runner,
    allow_retrain=False,
)
error_display = error_samples[[
    "test_index", "true_name", "predicted_name", "confidence_predicted_class",
    "original_token_count", "retained_token_count", "truncated_token_count",
    "unk_count_in_retained", "text",
]].copy()
error_display["text"] = error_display["text"].str.slice(0, 400) + "..."
display(error_display)
display(Image(filename=str(LAB_DIR / "error_attention_examples.png")))
print("Detailed report:", LAB_DIR / "error_analysis.md")